# sorted-computational-graph — faded example 2: Reverse topological order for the backward pass

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `sorted-computational-graph`. Running the beacon reports progress on the `Backprop: Sorted computation graph` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Sorted computation graph` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sorted-computational-graph`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sorted-computational-graph"
DD_SUBTOPIC = "Backprop: Sorted computation graph"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The standard DFS topological sort returns nodes in deps-first order (leaves first, end-node last). For the backward pass, you need the opposite: end-node first, leaves last. This is simply `[::-1]`. The critical property is that for every edge parent→child, the parent must appear before the child in backward order, which is exactly what the reversal gives.

## Faded exercise 2

Given `topological_sort` returns the forward (deps-first) order, produce the backward-pass order needed for reverse-mode autodiff.

1. Call `topological_sort(tensor, get_children)` to get the forward order.
2. Reverse it.

The blank step is reversing the topological sort result to get backward-pass order.

**Fill in:** Reverse the result of topological_sort so the end node comes first (backward-pass order).

In [ ]:
import torch as t

t.manual_seed(0)

class FakeRecipe:
    def __init__(self, parents):
        self.parents = {i: p for i, p in enumerate(parents)}

class FakeTensor:
    def __init__(self, name, parents=None):
        self.name = name
        self.recipe = FakeRecipe(parents) if parents else None

def topological_sort(node, get_children):
    result = []
    perm = set()
    def visit(cur):
        if id(cur) in perm:
            return
        perm.add(id(cur))
        for child in get_children(cur):
            visit(child)
        result.append(cur)
    visit(node)
    return result

def sorted_computational_graph(tensor):
    def get_children(n):
        if n.recipe is None:
            return []
        return list(n.recipe.parents.values())
    fwd = topological_sort(tensor, get_children)
    bwd = None  # TODO: Reverse the result of topological_sort so the end node comes first (backward-pass order).
    return bwd

a = FakeTensor('a')
b = FakeTensor('b', [a])
c = FakeTensor('c', [b])

bwd = sorted_computational_graph(c)
print('Backward order:', [n.name for n in bwd])  # ['c', 'b', 'a']
print('End node first:', bwd[0] is c)
print('Leaf last:', bwd[-1] is a)


def _test():
    class FakeRecipe:
        def __init__(self, parents):
            self.parents = {i: p for i, p in enumerate(parents)}
    class FakeTensor:
        def __init__(self, name, parents=None):
            self.name = name
            self.recipe = FakeRecipe(parents) if parents else None

    a = FakeTensor('a')
    b = FakeTensor('b', [a])
    c = FakeTensor('c', [b])

    bwd = sorted_computational_graph(c)
    assert len(bwd) == 3, f'len={len(bwd)}'
    assert bwd[0] is c, 'end node should be first'
    assert bwd[-1] is a, 'leaf should be last'
    assert bwd[1] is b


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(0)

class FakeRecipe:
    def __init__(self, parents):
        self.parents = {i: p for i, p in enumerate(parents)}

class FakeTensor:
    def __init__(self, name, parents=None):
        self.name = name
        self.recipe = FakeRecipe(parents) if parents else None

def topological_sort(node, get_children):
    result = []
    perm = set()
    def visit(cur):
        if id(cur) in perm:
            return
        perm.add(id(cur))
        for child in get_children(cur):
            visit(child)
        result.append(cur)
    visit(node)
    return result

def sorted_computational_graph(tensor):
    def get_children(n):
        if n.recipe is None:
            return []
        return list(n.recipe.parents.values())
    fwd = topological_sort(tensor, get_children)
    bwd = fwd[::-1]
    return bwd

a = FakeTensor('a')
b = FakeTensor('b', [a])
c = FakeTensor('c', [b])

bwd = sorted_computational_graph(c)
print('Backward order:', [n.name for n in bwd])
print('End node first:', bwd[0] is c)
print('Leaf last:', bwd[-1] is a)
```
</details>